# 02 · RAG Indexing
**Medical chatbot MVP**

This notebook:
1. Loads the processed documents from the phase 1
2. Embeds them with `text-embedding-3-small` model
3. Creates and saves FAISS index
4. Tests retrieval with example questions
5. Visualize embedding space with UMAP


In [ ]:
import sys
import json
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import faiss
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, str(Path('..').resolve()))

BASE_DIR = Path('..').resolve()
DOCUMENTS_PATH = BASE_DIR / 'data' / 'processed' / 'rag_documents.json'
INDEX_DIR = BASE_DIR / 'data' / 'processed' / 'faiss_index'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI()
print('OpenAI client OK')
print(f'Documents path: {DOCUMENTS_PATH}')
print(f'Index dir: {INDEX_DIR}')

In [ ]:
with open(DOCUMENTS_PATH, encoding='utf-8') as f:
    documents = json.load(f)

df = pd.DataFrame([{
    'id': d['id'],
    'type': d['metadata']['type'],
    'specialty': d['metadata'].get('specialty', '—'),
    'chars': len(d['text']),
} for d in documents])

print(f'Total documents: {len(documents)}')
print()
print(df.groupby('type').agg(count=('id','count'), avg_chars=('chars','mean')).round(0))

In [ ]:

for doc_type in ['doctor', 'specialty', 'knowledge_base']:
    sample = next(d for d in documents if d['metadata']['type'] == doc_type)
    print(f'=== {doc_type.upper()} ===')
    print(sample['text'][:400])
    print('...' if len(sample['text']) > 400 else '')
    print()

In [ ]:
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
BATCH_SIZE = 50

def embed_texts(texts: list[str]) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
        print(f'{min(i+BATCH_SIZE, len(texts))}/{len(texts)} embedded')
    return np.array(all_embeddings, dtype=np.float32)

texts = [d['text'] for d in documents]
embeddings = embed_texts(texts)
print(f'\Result: {embeddings.shape}  (documents X dimensions)')

In [ ]:

np.save(INDEX_DIR / 'embeddings_raw.npy', embeddings)

In [ ]:
emb_normalized = embeddings.copy()
faiss.normalize_L2(emb_normalized)

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(emb_normalized)

print(f'FAISS index: {index.ntotal} vectors, {EMBEDDING_DIM}D')
print(f'Type: {type(index).__name__}')

In [ ]:
metadata = [{'id': d['id'], **d['metadata']} for d in documents]

faiss.write_index(index, str(INDEX_DIR / 'index.faiss'))

with open(INDEX_DIR / 'metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

texts_map = {d['id']: d['text'] for d in documents}
with open(INDEX_DIR / 'texts.pkl', 'wb') as f:
    pickle.dump(texts_map, f)

for f in sorted(INDEX_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

In [ ]:
def search(query: str, top_k: int = 5, doc_type: str = None) -> pd.DataFrame:
    """A helper function for a retrieval test in the notebook."""
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    q_vec = np.array([response.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(q_vec)

    fetch_k = top_k * 5 if doc_type else top_k
    scores, indices = index.search(q_vec, min(fetch_k, index.ntotal))

    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        meta = metadata[idx]
        if doc_type and meta.get('type') != doc_type: continue
        rows.append({
            'score': round(float(score), 4),
            'type': meta.get('type'),
            'id': meta.get('id'),
            'specialty': meta.get('specialty', '—'),
        })
        if len(rows) >= top_k: break

    return pd.DataFrame(rows)

In [ ]:

query = 'Имам болки в гърдите и задух'
print(f'Query: "{query}"\n')
search(query, top_k=4)

In [ ]:

query = 'силно главоболие и изтръпване на ръцете'
print(f'Query: "{query}"\n')
search(query, top_k=4)

In [ ]:

query = 'кардиолог с опит в аритмии'
print(f'Query: "{query}" (само лекари)\n')
search(query, top_k=3, doc_type='doctor')

In [ ]:

query = 'до колко часа работи центърът'
print(f'Query: "{query}" (само knowledge base)\n')
search(query, top_k=3, doc_type='knowledge_base')

In [ ]:

query = 'има ли лекар, който приема по НЗОК'
print(f'Query: "{query}"\n')
search(query, top_k=5)

In [ ]:

query = 'внезапна парализа на лицето и загуба на реч'
print(f'Query: "{query}"\n')
search(query, top_k=4)

## Analysys for similarity scores


In [ ]:
test_queries = [
    ('болка в гърдите и задух', 'кардиолог'),
    ('главоболие и изтръпване', 'невролог'),
    ('болка в коляното след спорт', 'ортопед'),
    ('обрив и сърбеж по кожата', 'дерматолог'),
    ('киселини и стомашна болка', 'гастроентеролог'),
    ('умора и наддаване на тегло', 'ендокринолог'),
    ('хронична кашлица и задух', 'пулмолог'),
    ('болка в ухото и намален слух', 'УНГ лекар'),
    ('нередовен цикъл и болка в корема', 'гинеколог'),
    ('замъглено зрение', 'офталмолог'),
    ('парене при уриниране и кръв в урина','уролог'),
    ('депресия и безсъние', 'психиатър'),
]

results_summary = []
for query, expected in test_queries:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    q_vec = np.array([response.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, 1)
    top_meta = metadata[idxs[0][0]]
    top_score = float(scores[0][0])

    results_summary.append({
        'query': query[:40],
        'expected': expected,
        'retrieved': top_meta.get('specialty', top_meta.get('type','—')),
        'score': round(top_score, 4),
        'correct': expected.lower() in top_meta.get('specialty','').lower()
    })

df_results = pd.DataFrame(results_summary)
accuracy = df_results['correct'].mean()
print(f'Accuracy of top-1 retrieval: {accuracy:.0%} ({df_results["correct"].sum()}/{len(df_results)})')
print()
df_results

In [ ]:
fig = px.bar(
    df_results,
    x='query', y='score',
    color='correct',
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    title='Similarity score per test requests (top-1)',
    labels={'query': 'Request', 'score': 'Cosine similarity', 'correct': 'Correct?'},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

## Visualization of embedding dimension (UMAP)

We’re scaling down from 1536D to 2D to see how the documents are arranged.
If the RAG works well, documents from the same fields should be grouped together.


In [ ]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
    coords = reducer.fit_transform(emb_normalized)
    method = 'UMAP'
else:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=42)
    coords = reducer.fit_transform(emb_normalized)
    method = 'PCA'

print(f'Reduction with {method}: {emb_normalized.shape} → {coords.shape}')

In [ ]:
df_plot = pd.DataFrame({
    'x': coords[:, 0],
    'y': coords[:, 1],
    'type': [m.get('type') for m in metadata],
    'specialty': [m.get('specialty', m.get('source', m.get('type', '—'))) for m in metadata],
    'id': [m.get('id') for m in metadata],
})

fig = px.scatter(
    df_plot, x='x', y='y',
    color='specialty',
    symbol='type',
    hover_data=['id', 'type'],
    title=f'Embedding dimension ({method}) — {len(documents)} documents',
    width=900, height=600,
)
fig.update_traces(marker_size=8)
fig.show()

## Totals

In [ ]:
print('=' * 50)
print('RAG INDEXING SUMMARY')
print('=' * 50)
print(f'Model: {EMBEDDING_MODEL}')
print(f'Dimensions: {EMBEDDING_DIM}')
print(f'Documents: {index.ntotal}')
print(f'FAISS type: IndexFlatIP (cosine)')
print()
print('Files:')
for f in sorted(INDEX_DIR.iterdir()):
    print(f'  {f.name:<25} {f.stat().st_size/1024:.1f} KB')
print()
